# Does `hl.vds.split_multi` really relabel other ALTs as reference?

Run on the **All of Us Researcher Workbench**, Hail Genomic Analysis
environment. It is small and cheap: one multi-allelic site, a few hundred
samples.

## The claim under test

A VDS row can carry more than two alleles. `split_multi` breaks it into one
**bi-allelic** row per ALT, and to fit a genotype into a bi-allelic row it has
to rewrite it: **at each output row, any allele that is not *that row's* ALT is
relabeled as the reference.** That is *downcoding*.

If the claim is true, then for a row like

```
chr1:X   alleles = [REF, ALT1, ALT2]
```

a sample whose true genotype is `REF/ALT1` will, at the split row
`[REF, ALT2]`, report a genotype of **`0/0`** — identical to a sample that is
genuinely homozygous reference. The fact that it carries *some* non-reference
allele would be gone.

That matters because the score for a **REF**-effect allele is computed as
`2w - w * (ALT copies)`. If "ALT copies" is read off the downcoded genotype, a
REF/ALT1 carrier subtracts nothing and is credited **two** copies of the
reference when it has **one**. This is `TODO.md`, Finding 6.

## How this notebook could prove me wrong

Nothing below is asserted in advance. We print the sample's genotype
**before** the split and **after** it, at each output row, side by side. If the
`0/0` never appears, the claim is false and the fix was unnecessary.

In [ ]:
# If aoutools isn't already installed in this environment:
# !pip install -q git+https://github.com/dokyoonkimlab/aoutools.git@dev

import hail as hl
import pandas as pd

from aoutools import get_vds_path, init_hail

init_hail()

VDS_PATH = get_vds_path()
print("VDS:", VDS_PATH)

## Find a real tri-allelic site

We need a site with **two** ALT alleles that both have carriers -- otherwise
there is no "other ALT" to be relabeled and nothing to see.

In [ ]:
WINDOW = "chr1:1000000-2000000"
N_SAMPLES = 300

vds = hl.vds.read_vds(VDS_PATH)
vds = hl.vds.filter_intervals(
    vds,
    [hl.parse_locus_interval(WINDOW, reference_genome="GRCh38")],
    keep=True,
)

sample_ids = vds.variant_data.cols().head(N_SAMPLES).s.collect()
samples_ht = hl.Table.parallelize(
    [{"s": s} for s in sample_ids], hl.tstruct(s=hl.tstr)
).key_by("s")
vds = hl.vds.filter_samples(vds, samples_ht)

vd = vds.variant_data
# LGT is a *local* call: it indexes into LA, not into `alleles`. lgt_to_gt
# converts it to a global call, whose indices line up with `alleles`.
gt = hl.vds.lgt_to_gt(vd.LGT, vd.LA)
vd = vd.annotate_rows(
    n_alt1=hl.agg.count_where((gt[0] == 1) | (gt[1] == 1)),
    n_alt2=hl.agg.count_where((gt[0] == 2) | (gt[1] == 2)),
)
rows = vd.rows()
rows = rows.filter(
    (hl.len(rows.alleles) == 3) & (rows.n_alt1 > 0) & (rows.n_alt2 > 0)
)

site = rows.head(1).collect()
assert site, "no tri-allelic site with carriers of both ALTs -- widen WINDOW"
site = site[0]

REF, ALT1, ALT2 = site.alleles
print(f"site   : {site.locus}")
print(f"alleles: [{REF}, {ALT1}, {ALT2}]   (index 0 = REF, 1 = ALT1, 2 = ALT2)")
print(f"carriers of ALT1 ({ALT1}): {site.n_alt1}")
print(f"carriers of ALT2 ({ALT2}): {site.n_alt2}")

## Cut down to that one site

In [ ]:
locus_ht = hl.Table.parallelize(
    [{"locus": site.locus}], hl.tstruct(locus=hl.tlocus("GRCh38"))
).key_by("locus")
intervals = locus_ht.select(
    interval=hl.interval(locus_ht.locus, locus_ht.locus, includes_end=True)
).key_by("interval")

vds_site = hl.vds.filter_intervals(vds, intervals, keep=True)
print("rows kept:", vds_site.variant_data.count_rows())

## BEFORE the split -- each sample's true genotype

This is the original, multi-allelic row. Genotypes are global indices into
`[REF, ALT1, ALT2]`.

We also record `n_non_ref` = `LGT.n_alt_alleles()`, the sample's **total**
number of non-reference alleles at this site. Hold on to that number: it is the
one downcoding is about to destroy.

In [ ]:
vd = vds_site.variant_data
gt = hl.vds.lgt_to_gt(vd.LGT, vd.LA)

before = vd.select_entries(
    genotype=hl.str(gt),
    n_non_ref=vd.LGT.n_alt_alleles(),
).entries()
before = before.select("genotype", "n_non_ref").collect()


def indices(gt_str):
    """Allele indices of a hail call string, sorted, phase discarded.

    AoU calls are PHASED, so hail renders them '0|2', not '0/2'. Phase is
    irrelevant to a copy count, and if it leaks into a string comparison it
    reads as a different genotype: '0|0' != '0/0' would make the downcoding
    below look absent when it is not. So every genotype in this notebook is
    normalized here, and nowhere else.
    """
    if gt_str is None:
        return None
    return sorted(int(i) for i in gt_str.replace("|", "/").split("/"))


def unphased(gt_str):
    """'2|0' -> '0/2'."""
    idx = indices(gt_str)
    return None if idx is None else "/".join(str(i) for i in idx)


def spell(gt_str):
    """'0|1' -> 'C/G', using this site's allele list."""
    idx = indices(gt_str)
    if idx is None:
        return "no-call"
    return "/".join(site.alleles[i] for i in idx)


before_df = pd.DataFrame(
    [
        {
            "sample": r.s,
            "true_genotype": unphased(r.genotype),
            "true_alleles": spell(r.genotype),
            "n_non_ref": r.n_non_ref,
        }
        for r in before
    ]
)
print(
    f"{len(before_df)} samples have an entry at this site "
    f"(the rest are hom-ref and have NO entry at all)\n"
)
print(before_df.true_genotype.value_counts().to_string())
before_df.head(10)

## AFTER the split -- the same samples, at each output row

`split_multi` turns that one row into **two** bi-allelic rows:

* `[REF, ALT1]` — "is it an ALT1?"
* `[REF, ALT2]` — "is it an ALT2?"

Every sample that had an entry gets one in **both**. Watch what its genotype
becomes.

In [ ]:
split_vd = hl.vds.split_multi(vds_site).variant_data

after = split_vd.select_entries(genotype=hl.str(split_vd.GT)).entries()
# `alleles` is a key field, so it comes along automatically --
# selecting it explicitly is an error.
after = after.collect()

after_df = pd.DataFrame(
    [
        {
            "sample": r.s,
            "split_row": f"[{r.alleles[0]}, {r.alleles[1]}]",
            # split_multi preserves phase, so normalize this side too.
            "split_genotype": unphased(r.genotype),
        }
        for r in after
    ]
)

wide = after_df.pivot(
    index="sample", columns="split_row", values="split_genotype"
).reset_index()
combined = before_df.merge(wide, on="sample")
combined = combined[
    ["sample", "true_alleles", "true_genotype", "n_non_ref"]
    + sorted(c for c in wide.columns if c != "sample")
]
combined.head(15)

## The verdict

Look at the rows whose `true_alleles` is **REF/ALT1**, and read across to the
`[REF, ALT2]` column.

If the claim is right, that column says **`0/0`** — the ALT1 has been relabeled
as reference, because it is not *this row's* ALT. The sample is now
indistinguishable from a genuine homozygous-reference sample, even though
`n_non_ref` (recorded before the split) still says it carries one
non-reference allele.

The cell below decides it, and does not assume the answer.

In [ ]:
row_alt1 = f"[{REF}, {ALT1}]"
row_alt2 = f"[{REF}, {ALT2}]"

# Samples carrying ALT1 but NOT ALT2 -- these are the ones at risk.
carriers = combined[
    combined.true_alleles.isin([f"{REF}/{ALT1}", f"{ALT1}/{ALT1}"])
]

print(f"{len(carriers)} samples carry {ALT1} and not {ALT2}\n")
print(
    carriers[["sample", "true_alleles", "n_non_ref", row_alt1, row_alt2]]
    .head(10)
    .to_string(index=False)
)

downcoded = (carriers[row_alt2] == "0/0").all()
print()
if downcoded and len(carriers):
    print(
        f"CONFIRMED. At the {row_alt2} row every {ALT1} carrier reports "
        f"0/0 -- looking exactly like a hom-ref sample -- while n_non_ref, "
        f"taken before the split, still knows it carries a non-reference "
        f"allele.\n\n"
        f"So `GT.n_alt_alleles()` at that row is 0, and a REF-effect score of "
        f"2w - w*0 = 2w credits these samples with TWO copies of {REF}. A "
        f"{REF}/{ALT1} sample has ONE. A {ALT1}/{ALT1} sample has NONE.\n\n"
        f"This is why the fix subtracts n_non_ref instead: "
        f"2 - n_non_ref gives the right answer for both."
    )
else:
    print(
        "NOT confirmed -- the ALT1 carriers do not report 0/0 at the ALT2 "
        "row. The downcoding claim is false for this VDS, and Finding 6's "
        "premise does not hold. Inspect `combined` above."
    )

## And the ALT-effect case, for contrast

Downcoding is only *wrong* for a REF-effect allele. If the effect allele is
**ALT2**, then a sample carrying ALT1 genuinely has **zero** copies of it --
and `0/0` at the `[REF, ALT2]` row says exactly that. Correct.

That is why the fix does not simply swap one dosage for the other everywhere:
the two orientations must count different things.

In [ ]:
check = carriers[[row_alt2]].copy()
check["copies_of_ALT2_reported"] = check[row_alt2].map(
    lambda g: 0 if g is None else g.count("1")
)
print(
    f"copies of {ALT2} reported for {ALT1}-carriers at the {row_alt2} row: "
    f"{sorted(check.copies_of_ALT2_reported.unique())}"
)
print(
    f"\nCorrect: they carry no {ALT2}. For an ALT-effect weight the "
    f"downcoded genotype is exactly right, and only the REF-effect case "
    f"needs n_non_ref."
)